# 02 — Data Preparation

Stage 3: build the shared feature pipeline that Forecasting, Inventory, and Pricing all consume, and produce the chronological train/val/test split reused across every later stage.

Scope is controlled by `src/config.py:N_SERIES_SUBSET` (currently a subset, top-N series by volume, per the plan's "start small then scale" decision). Re-running this notebook after setting it to `"all"` regenerates the identical pipeline on all 100 series.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config
from src.data.load import read_cleaned
from src.data.features import select_series, engineer_features, build_features_full
from src.forecasting.splits import time_split

pd.set_option("display.max_columns", 60)
print("N_SERIES_SUBSET =", config.N_SERIES_SUBSET)


N_SERIES_SUBSET = 10


## 1. Scope selection

Filter the cleaned panel down to the configured subset of (Store ID, Product ID) series, chosen by total historical volume.

In [2]:
cleaned = read_cleaned()
scoped = select_series(cleaned, config.N_SERIES_SUBSET)

n_series = scoped.groupby(config.ID_COLS, observed=True).ngroups
print(f"scoped to {n_series} series, {len(scoped)} rows")
scoped.groupby(config.ID_COLS, observed=True)["Units Sold"].sum().sort_values(ascending=False)


scoped to 10 series, 7310 rows


Store ID  Product ID
S005      P0015         109099
S003      P0013         107479
S002      P0001         105999
S005      P0003         105667
S003      P0005         105408
S002      P0009         105376
          P0020         105343
S003      P0014         105188
S004      P0016         104777
S003      P0017         104566
Name: Units Sold, dtype: int64

## 2. Feature engineering

Apply the shared feature set (calendar, lag, rolling, price, inventory, one-hot categoricals) via `src/data/features.py`.

In [3]:
features = engineer_features(scoped)
print(features.shape)
features.head()


(7310, 55)


,Date,Store ID,Product ID,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Holiday/Promotion,Competitor Pricing,day_of_week,is_weekend,month,week_of_year,day_of_year,dow_sin,dow_cos,month_sin,month_cos,units_sold_lag_1,units_sold_lag_7,units_sold_lag_14,units_sold_lag_28,units_sold_roll_mean_7,units_sold_roll_std_7,units_sold_roll_mean_14,units_sold_roll_std_14,units_sold_roll_mean_28,units_sold_roll_std_28,inventory_roll_mean_7,discount_roll_mean_7,price_gap,price_gap_pct,discount_pct,effective_price,days_of_supply,stockout_flag,Category_Clothing,Category_Electronics,Category_Furniture,Category_Groceries,Category_Toys,Region_East,Region_North,Region_South,Region_West,Weather Condition_Cloudy,Weather Condition_Rainy,Weather Condition_Snowy,Weather Condition_Sunny,Seasonality_Autumn,Seasonality_Spring,Seasonality_Summer,Seasonality_Winter
0,2022-01-01,S002,P0001,343,104,144,112.55,32.80,20,1,30.78,5,1,1,52,1,-0.974928,-0.222521,0.5,0.866025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.02,0.065627,0.20,26.2400,NaN,0,False,False,False,True,False,False,False,True,False,False,False,False,True,False,True,False,False
1,2022-01-02,S002,P0001,402,343,141,339.04,66.57,20,1,63.92,6,1,1,52,2,-0.781831,0.623490,0.5,0.866025,104.0,NaN,NaN,NaN,104.0,NaN,104.0,NaN,104.0,NaN,343.000000,20.000000,2.65,0.041458,0.20,53.2560,3.865385,0,False,True,False,False,False,False,False,False,True,False,True,False,False,False,False,True,False
2,2022-01-03,S002,P0001,405,96,171,100.82,48.08,0,1,49.40,0,0,1,1,3,0.000000,1.000000,0.5,0.866025,343.0,NaN,NaN,NaN,223.5,168.998521,223.5,168.998521,223.5,168.998521,372.500000,20.000000,-1.32,-0.026721,0.00,48.0800,1.812081,0,False,False,False,False,True,False,True,False,False,False,True,False,False,True,False,False,False
3,2022-01-04,S002,P0001,108,83,25,85.50,45.49,5,1,44.33,1,0,1,1,4,0.781831,0.623490,0.5,0.866025,96.0,NaN,NaN,NaN,181.0,140.353126,181.0,140.353126,181.0,140.353126,383.333333,13.333333,1.16,0.026167,0.05,43.2155,0.596685,0,True,False,False,False,False,False,True,False,False,False,False,False,True,False,True,False,False
4,2022-01-05,S002,P0001,250,161,70,171.45,56.24,15,1,60.32,2,0,1,1,5,0.974928,-0.222521,0.5,0.866025,83.0,NaN,NaN,NaN,156.5,124.634131,156.5,124.634131,156.5,124.634131,314.500000,11.250000,-4.08,-0.067639,0.15,47.8040,1.597444,0,False,False,True,False,False,True,False,False,False,False,False,True,False,False,True,False,False


In [4]:
# sanity check: lag/rolling features should be NaN only at the start of each
# series (no leakage, no unexpected gaps mid-series)
na_counts = features[[c for c in features.columns if "lag" in c or "roll" in c]].isna().sum()
na_counts


units_sold_lag_1            10
units_sold_lag_7            70
units_sold_lag_14          140
units_sold_lag_28          280
units_sold_roll_mean_7      10
units_sold_roll_std_7       20
units_sold_roll_mean_14     10
units_sold_roll_std_14      20
units_sold_roll_mean_28     10
units_sold_roll_std_28      20
inventory_roll_mean_7       10
discount_roll_mean_7        10
stockout_flag                0
dtype: int64

## 3. Save engineered features

Persist the full feature set (`data/processed/features_full.parquet`), reused directly by this notebook and by every downstream stage — regenerated here via `build_features_full()` so the saved file always matches this notebook's logic exactly.

In [5]:
features = build_features_full()
print("saved:", config.FEATURES_FULL, features.shape)


saved: /Users/athens/Downloads/Coding/Retail Store Inventory/data/processed/features_full.parquet (7310, 55)


## 4. Chronological train/val/test split

Identical cut dates for every series, no shuffling:

- Train: 2022-01-01 -> 2023-06-30
- Val: 2023-07-01 -> 2023-09-30
- Test: 2023-10-01 -> 2024-01-01

This exact split is reused unmodified by the Inventory simulation and Pricing evaluation stages so all three challenges are judged on the same held-out window.

In [6]:
train, val, test = time_split(features)

for name, part in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s} rows={len(part):5d}  {part[config.DATE_COL].min().date()} -> {part[config.DATE_COL].max().date()}")

assert len(train) + len(val) + len(test) == len(features), "split must cover every row exactly once"


train rows= 5460  2022-01-01 -> 2023-06-30
val   rows=  920  2023-07-01 -> 2023-09-30
test  rows=  930  2023-10-01 -> 2024-01-01


In [7]:
train_dates = set(train[config.DATE_COL])
val_dates = set(val[config.DATE_COL])
test_dates = set(test[config.DATE_COL])
assert train_dates.isdisjoint(val_dates)
assert val_dates.isdisjoint(test_dates)
assert train_dates.isdisjoint(test_dates)
print("no overlap between train/val/test date ranges - OK")


no overlap between train/val/test date ranges - OK


In [8]:
config.DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
train.to_parquet(config.DATA_PROCESSED_DIR / "train.parquet", index=False)
val.to_parquet(config.DATA_PROCESSED_DIR / "val.parquet", index=False)
test.to_parquet(config.DATA_PROCESSED_DIR / "test.parquet", index=False)
print("saved train/val/test parquet files to", config.DATA_PROCESSED_DIR)


saved train/val/test parquet files to /Users/athens/Downloads/Coding/Retail Store Inventory/data/processed


## Stage 3 summary

- Feature pipeline (`src/data/features.py`) applies calendar, lag (1/7/14/28d), rolling (7/14/28d, leakage-safe via shift), price, and inventory features, plus one-hot encoding, identically for every downstream challenge.
- Scope is controlled by a single config flag (`N_SERIES_SUBSET`), currently a volume-based subset; switching it to `"all"` reruns this exact notebook on all 100 series with no code changes.
- Chronological train/val/test split verified to cover every row exactly once with zero date overlap.
- Outputs saved: `features_full.parquet`, `train.parquet`, `val.parquet`, `test.parquet` in `data/processed/` — ready for Stage 4 (Forecasting, Inventory, Pricing).